# Hero Analysis Notebook

Verifying the flattened Hero tables using rich visualization.

In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe
# Import shared warehouse logic
from super.core.warehouse import get_hero_genome_summary, get_all_heroes_data


In [ ]:

# Bootstrap & Session
bootstrap_spark_env()
spark = SparkSession.builder.appName("HeroAnalyzer").getOrCreate()

# Config
conf = utils.get_app_conf("generate_powers")
warehouse_root = os.path.join(conf.get_string("stage_root"), "warehouse")
print(f"Reading Warehouse: {warehouse_root}")

## 1. Hero Genome Reports (Unified View - Spark)
Analysis of the **Gene Cluster** for each hero using Spark.

In [ ]:
# Use shared logic for DRY compliance
full_report = get_hero_genome_summary(spark, warehouse_root)

# Display as a Form-like table
display_scrollable_dataframe(full_report.orderBy("hero_name").toPandas())

## 2. Deep Dive: Bugs Bunny's Cluster (Spark)

In [ ]:
bugs_report = full_report.filter(F.col("hero_name") == "Bugs Bunny")
# Show full text for Bugs to verify richness
for row in bugs_report.collect():
    print(f"HERO: {row['hero_name']} ({row['ontology']})")
    print(f"BIO: {row['bio']}")
    print(f"MASTER REGULATOR: {row['master_regulator_id']} ({row['mutation_class']})")
    print(f"CLUSTER SIZE: {row['cluster_size']} regulated genes")
    print(f"CLUSTER MAP: {row['cluster_network_summary']}")


## 3. Interactive Ontology View (Pandas)
View the hero table filtered by ontology using **Pandas** directly.

In [ ]:
# 1. List Available Ontologies
profiles_path = os.path.join(warehouse_root, "hero_profiles")
partitions = [d for d in os.listdir(profiles_path) if d.startswith("ontology=")]
print("Available Ontologies:")
for p in partitions:
    print(f" - {p.replace('ontology=', '')}")

In [ ]:
# 2. Configure Ontology
# CHANGE THIS VALUE to filter by a different universe (e.g. "Marvel", "DC Comics", "Looney Tunes")
SELECTED_ONTOLOGY = "Looney Tunes"

print(f"Loading data for: {SELECTED_ONTOLOGY}...")
pandas_report = get_all_heroes_data(warehouse_root=warehouse_root, ontology=SELECTED_ONTOLOGY)

display_scrollable_dataframe(pandas_report)